# Lesson 7 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent. You will use `RemoteA2aAgent` (which acts as A2A Client) to connect to the servers you started in previous lessons.

![Sequential Workflow](sequential.png)

## 7.1. Start the A2A Servers

First, ensure that your `PolicyAgent` server is running.
- Open Terminal 1 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_policy_agent.py` (you don't need to go back to the previous lesson).

In [1]:
import os

from IPython.display import IFrame

url = os.environ.get("DLAI_LOCAL_URL").format(port=8888)
# Terminal 1: uv run a2a_policy_agent.py
IFrame(f"{url}terminals/1", width=800, height=200)

Second, ensure that your Research Agent is running.
- Open Terminal 2 by running the cell below.
- If the agent is still running from the previous lesson, you don't need to do anything.
- If the agent has stopped, type: `uv run a2a_research_agent.py` (you don't need to go back to the previous lesson).

In [2]:
# Terminal 2: uv run a2a_research_agent.py
IFrame(f"{url}terminals/2", width=800, height=200)

## 7.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as A2A clients that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [3]:
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner

import logging
import warnings

logging.disable(level=logging.WARNING)
warnings.filterwarnings("ignore")

In [4]:
load_dotenv()
host = os.environ.get("AGENT_HOST")
policy_port = os.environ.get("POLICY_AGENT_PORT")
research_port = os.environ.get("RESEARCH_AGENT_PORT")

In [5]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [6]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [7]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")

	ℹ️ root_agent initialized


## 7.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.

In [8]:
prompt = "How can I get mental health therapy?"

**Note:** It takes a few seconds for the output to display.

In [9]:
print("Running Healthcare Workflow Agent")

runner = InMemoryRunner(root_agent)

for event in await runner.run_debug(prompt, quiet=True):
    if event.is_final_response() and event.content:
        display(Markdown(event.content.parts[0].text))

Running Healthcare Workflow Agent


Getting started with mental health therapy can feel overwhelming, but breaking the process down into manageable steps makes it much easier. Based on current healthcare resources, here is a comprehensive guide to finding and accessing mental health therapy.

### 🚨 Immediate Crisis Support
If you or someone you know is in a life-threatening situation or experiencing immediate emotional distress, please use these free, 24/7 resources:
* **988 Suicide & Crisis Lifeline**: Call or text **988**, or chat online for confidential support. Veterans can also use this line to reach the Veterans Crisis Line.
* **Crisis Text Line**: Text **"HOME" to 741741** to connect with a trained crisis counselor.
* **Childhelp National Child Abuse Hotline**: Call **1-800-4-A-CHILD** (1-800-422-4453) for child and adult survivors of abuse. 
* **Dial 211**: In many areas, this will connect you with immediate local mental health crisis services.
* **Call 911** or go to your nearest emergency room.
*(Sources: National Institutes of Health, Mental Health America)*

---

### Step 1: Define Your Needs and Goals
You do not need an exact diagnosis to start therapy, but it helps to know what you want to work on. Before searching, ask yourself:
* What challenges am I facing? (e.g., anxiety, sleep issues, grief, relationship problems, life transitions, trauma, or anger management)
* Do I want one-on-one therapy, couples counseling, or group therapy?
* Do I prefer in-person sessions or virtual visits (telehealth)?
* Do I need a therapist with a specific cultural background or identity who might better understand my lived experiences?
*(Sources: Psychology Today, University Hospitals, Healthy Minds NYC)*

### Step 2: Understand the Types of Mental Health Providers
"Therapist" is a broad term covering several credentials. Knowing the difference helps you find the right level of care:
* **Psychiatrists (MD or DO)**: Medical doctors who specialize in mental health. They can diagnose conditions and prescribe medication.
* **Psychologists (PhD or PsyD)**: Professionals with doctoral degrees who provide talk therapy and can conduct psychological testing and assessments.
* **Licensed Clinical Social Workers (LCSWs)**: Therapists trained to address emotional and behavioral issues within the context of a person's social environment.
* **Licensed Professional Counselors (LPCs) / Licensed Mental Health Counselors (LMHCs)**: Professionals who focus on talk therapy for a wide range of mental health concerns.
*(Sources: Pearl Health, Psychology Today)*

### Step 3: Find a Provider
There are many avenues to locate a qualified mental health professional:
* **Your Primary Care Provider (PCP)**: Your doctor can perform an initial mental health screening and refer you to a trusted psychologist, social worker, or psychiatrist.
* **Your Health Insurance**: If you have insurance, call the number on the back of your card or use their online portal/app to find a list of in-network providers. This will help you understand your coverage limits and copays.
* **Online Directories**: You can filter searches by zip code, specialty, accepted insurance, and therapy approach using databases like:
    * *Psychology Today* Therapy Directory
    * American Psychological Association (APA) *Psychologist Locator*
    * *National Register of Health Service Psychologists*
    * *Theravive*
* **Federal and State Resources**: 
    * The **Substance Abuse and Mental Health Services Administration (SAMHSA)** offers a helpline and an online treatment locator.
    * **Medicare Physician Compare** and your **State Medicaid Office** can help you find providers who accept Medicare or Medicaid.
    * Your local health services department website is also a great starting point.
* **Schools and Universities**: If you are a student, check your college or university's health center for free or discounted peer support and counseling services.
* **Telehealth Platforms**: Online services like MDLive, BetterHelp, or Talkspace offer virtual access to therapists and psychiatrists.
*(Sources: National Institutes of Health, Mental Health America, Healthline, American Psychological Association)*

### Step 4: Evaluate and Interview Potential Therapists
Finding a therapist is highly personal, and it is completely normal to "shop around" until you find a good fit. Treat the initial contact like an interview:
* **Check Credentials**: Ensure the therapist is licensed to practice in your state.
* **Ask About Experience**: Ask if they have experience dealing with your specific concerns (e.g., depression, panic attacks, family issues).
* **Inquire About Evidence-Based Treatments**: Ask if they use scientifically tested treatments, such as Cognitive-Behavioral Therapy (CBT), psychodynamic therapy, or mindfulness-based therapy.
* **Schedule a Consultation**: Many therapists offer a free 10-to-15-minute introductory phone call. Use this time to gauge their personality and communication style. Feeling comfortable, safe, and genuinely heard by your provider is the most important factor in successful therapy.
*(Sources: American Psychological Association, Healthy Minds NYC, University Hospitals, Pearl Health)*

Based on the insurance document provided, here's how you can access mental health therapy under this plan:

## Coverage for Mental Health Services

This plan covers **outpatient and inpatient mental health, behavioral health, and substance abuse services**.

### Out-of-Network Provider Costs:
- **Office visits**: 30% coinsurance
- **Other outpatient services**: 30% coinsurance
- **Inpatient services**: 30% coinsurance (with 30% coinsurance for inpatient physician fees)

### In-Network Provider Costs (Lower cost):
- **Office visits**: 10% coinsurance
- **Other outpatient services**: 10% coinsurance
- **Inpatient services**: 10% coinsurance (with 10% coinsurance for inpatient physician fees)

### Important Details:
- These costs apply **after you meet your deductible** (\\$1,700 individual / \\$3,400 family for in-network)
- **No referral is required** to see a mental health specialist
- You can find **in-network providers** by contacting:
  - **Call**: (855) 431-5540
  - **Website**: includehealth.com/google

### Network Options:
- **NY residents** (excluding Suffolk County): NY Blue Access PPO network
- **GA residents**: GA Blue Open Access POS network
- **All other areas**: Blue Card PPO network

**For more details** about mental health coverage, see the complete plan document at go/benefitdocuments or contact Anthem Blue Cross directly.

## 7.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/5_ADKSequentialAgent.ipynb)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>
